# Retail Sales Analysis (Sample Superstore Subset)

This notebook analyzes a curated subset of retail sales data. It demonstrates data cleaning, transformation, exploratory analysis, and business insights with clear, refactored functions and readable variable names.


In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


In [ ]:
# Configuration
DATA_PATH = "data/retail_sales_sample.csv"


## Helper Functions


In [ ]:
def load_sales_data(file_path: str) -> pd.DataFrame:
    '''Load raw sales data from a CSV file.'''
    return pd.read_csv(file_path)


def clean_sales_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    '''Clean raw sales data and return a standardized DataFrame.'''
    cleaned_df = raw_df.copy()

    # Standardize text fields by trimming whitespace
    text_columns = [
        "customer_name",
        "segment",
        "region",
        "state",
        "category",
        "sub_category",
        "product",
    ]
    for column in text_columns:
        cleaned_df[column] = cleaned_df[column].astype(str).str.strip()

    # Normalize categorical casing
    cleaned_df["segment"] = cleaned_df["segment"].str.title()
    cleaned_df["region"] = cleaned_df["region"].str.title()
    cleaned_df["category"] = cleaned_df["category"].str.title()
    cleaned_df["sub_category"] = cleaned_df["sub_category"].str.title()

    # Parse dates safely (invalid formats become NaT)
    cleaned_df["order_date"] = pd.to_datetime(cleaned_df["order_date"], errors="coerce")
    cleaned_df["ship_date"] = pd.to_datetime(cleaned_df["ship_date"], errors="coerce")

    # Clean sales values
    cleaned_df["sales"] = (
        cleaned_df["sales"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    cleaned_df["sales"] = pd.to_numeric(cleaned_df["sales"], errors="coerce")

    # Clean discount values (support percent strings like "10%")
    def parse_discount(value) -> float:
        if pd.isna(value):
            return np.nan
        value_str = str(value).strip()
        if value_str.endswith("%"):
            return float(value_str.replace("%", "")) / 100
        return float(value_str)

    cleaned_df["discount"] = cleaned_df["discount"].apply(parse_discount)
    cleaned_df["discount"] = cleaned_df["discount"].fillna(0)

    # Convert numeric fields
    cleaned_df["quantity"] = pd.to_numeric(cleaned_df["quantity"], errors="coerce")
    cleaned_df["profit"] = pd.to_numeric(cleaned_df["profit"], errors="coerce")

    # Remove exact duplicate rows
    cleaned_df = cleaned_df.drop_duplicates()

    return cleaned_df


def add_features(cleaned_df: pd.DataFrame) -> pd.DataFrame:
    '''Add derived features for analysis.'''
    enriched_df = cleaned_df.copy()

    enriched_df["order_month"] = enriched_df["order_date"].dt.to_period("M").astype(str)
    enriched_df["ship_days"] = (enriched_df["ship_date"] - enriched_df["order_date"]).dt.days
    enriched_df["profit_margin"] = enriched_df["profit"] / enriched_df["sales"]

    # Bucket sales into small/medium/large orders
    enriched_df["sales_bucket"] = pd.cut(
        enriched_df["sales"],
        bins=[0, 50, 200, 1000],
        labels=["Small", "Medium", "Large"],
        include_lowest=True,
    )

    return enriched_df


def summarize_by_category(enriched_df: pd.DataFrame) -> pd.DataFrame:
    '''Summarize sales and profit by category.'''
    return (
        enriched_df.groupby("category")[["sales", "profit"]]
        .sum()
        .sort_values("sales", ascending=False)
    )


def summarize_by_region(enriched_df: pd.DataFrame) -> pd.Series:
    '''Summarize sales by region.'''
    return enriched_df.groupby("region")["sales"].sum().sort_values(ascending=False)


def summarize_by_segment(enriched_df: pd.DataFrame) -> pd.DataFrame:
    '''Summarize sales and profit by segment.'''
    return (
        enriched_df.groupby("segment")[["sales", "profit"]]
        .sum()
        .sort_values("sales", ascending=False)
    )


def top_customers_by_sales(enriched_df: pd.DataFrame, top_n: int = 10) -> pd.Series:
    '''Return the top N customers by total sales.'''
    return (
        enriched_df.groupby(["customer_id", "customer_name"])["sales"]
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
    )


def monthly_sales_trend(enriched_df: pd.DataFrame) -> pd.Series:
    '''Return total sales by order month.'''
    return enriched_df.groupby("order_month")["sales"].sum()


## Load Data


In [ ]:
raw_sales_df = load_sales_data(DATA_PATH)
raw_sales_df.head()


## Data Cleaning


In [ ]:
clean_sales_df = clean_sales_data(raw_sales_df)
clean_sales_df.isna().sum()


## Data Transformation


In [ ]:
analysis_df = add_features(clean_sales_df)
analysis_df.head()


## Exploratory Analysis


In [ ]:
analysis_df[["sales", "quantity", "discount", "profit", "ship_days"]].describe()


In [ ]:
category_summary = summarize_by_category(analysis_df)
category_summary


In [ ]:
region_summary = summarize_by_region(analysis_df)
region_summary


## Business Questions


In [ ]:
# Q1: Which categories generate the most sales and profit?
category_summary


In [ ]:
# Q2: Do higher discounts reduce profit margins?
discount_profit_correlation = analysis_df[["discount", "profit_margin"]].corr().loc["discount", "profit_margin"]
discount_profit_correlation


In [ ]:
# Q3: Which customer segments are most valuable?
segment_summary = summarize_by_segment(analysis_df)
segment_summary


In [ ]:
# Q4: How do sales trend over time?
monthly_sales = monthly_sales_trend(analysis_df)
monthly_sales


In [ ]:
# Q5: Who are the top customers by sales?
top_customers = top_customers_by_sales(analysis_df, top_n=10)
top_customers


## Visualizations


In [ ]:
# Sales by category (bar chart)
category_summary.plot(kind="bar", y="sales", legend=False)
plt.title("Total Sales by Category")
plt.ylabel("Sales")
plt.xlabel("Category")
plt.show()


In [ ]:
# Monthly sales trend (line chart)
monthly_sales.plot(marker="o")
plt.title("Monthly Sales Trend")
plt.ylabel("Sales")
plt.xlabel("Order Month")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Discount vs Profit Margin (scatter plot)
sns.scatterplot(data=analysis_df, x="discount", y="profit_margin", hue="category")
plt.title("Discount vs Profit Margin")
plt.show()


In [ ]:
# Sales distribution by segment (box plot)
sns.boxplot(data=analysis_df, x="segment", y="sales")
plt.title("Sales Distribution by Segment")
plt.show()


## Final Insights

- **Technology** drives the highest sales and strong profits, largely from smartphones and accessories.
- **Discounting shows a negative relationship with profit margin**, suggesting aggressive discounts erode profitability.
- **Consumer and Corporate segments contribute most of the revenue**, while Home Office is smaller but steady.
- **Sales trend upward into March**, indicating early-year momentum.
- A small set of customers account for a large share of revenue, which suggests opportunities for targeted retention campaigns.


## How To Run

1. Ensure Python 3.9+ is installed.
2. Install dependencies:
   - `python3 -m pip install pandas numpy matplotlib seaborn jupyter`
3. Run the notebook:
   - `python3 -m nbconvert --execute --to notebook --inplace sales_analysis.ipynb`
4. Open the notebook in Jupyter or VS Code to review the outputs and plots.

If you add new rows to the CSV, rerun the notebook to refresh all analyses and visuals.
